In [1]:
import torch
import torch.nn as nn
from torch.nn.utils.rnn import pad_sequence
from torch.utils.data import Dataset, DataLoader

import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.metrics import matthews_corrcoef,accuracy_score, roc_auc_score,precision_score,f1_score,confusion_matrix,roc_curve
from sklearn.model_selection import StratifiedKFold

from collections import defaultdict
import json
import random
import os
from typing import List, Tuple, Dict, Optional
import math
import re
import copy
import optuna
import gc

from warnings import filterwarnings
# Silence some expected warnings
filterwarnings("ignore")

In [2]:
# ============ 1. Tokenizer ============
smiles_token_pattern = (
    r"(\[[^\]]+]"           # bracket atoms/groups
    r"|Br?|Cl?"             # halogens
    r"|N|O|S|P|F|I|b|c|n|o|s|p"
    r"|\(|\)|\."            # punctuation
    r"|=|#|-|\+|\\|/"
    r"|:|~|@@|@"            # chirality
    r"|\?|>>?"
    r"|\*|\$"
    r"|\%[0-9]{2}"          # %10 etc.
    r"|[0-9])"
)
tokenizer_re = re.compile(smiles_token_pattern)

def tokenize(smiles: str):
    if smiles is None:
        return []
    s = str(smiles).strip()
    if not s:
        return []
    return tokenizer_re.findall(s)

In [3]:
# ============ 2. Vectorization ============
def smiles_to_ids(smiles, vocab, unk_token="<UNK>", bos_token=None, eos_token=None):
    toks = tokenize(smiles)
    ids = [vocab.get(t, vocab[unk_token]) for t in toks]
    if bos_token is not None:
        ids = [vocab[bos_token]] + ids
    if eos_token is not None:
        ids = ids + [vocab[eos_token]]
    return torch.tensor(ids, dtype=torch.long)

def collate_smiles_to_batch(smiles_batch, vocab, pad_token="<PAD>", bos_token=None, eos_token=None):
    seqs = [smiles_to_ids(s, vocab, bos_token=bos_token, eos_token=eos_token) for s in smiles_batch]
    padded = pad_sequence(seqs, batch_first=True, padding_value=vocab[pad_token])
    pad_mask = (padded == vocab[pad_token]).to(torch.bool)
    return padded, pad_mask

In [4]:
# ============ 3. Create Dataset ============
class SMILESDataset(Dataset):
    def __init__(self, df):
        self.smiles = ["" if pd.isna(x) else str(x) for x in df["smiles"].tolist()]
        self.compound_id = df["tcm_id"].tolist()

    def __getitem__(self, idx):
        return self.smiles[idx], self.compound_id[idx]

    def __len__(self):
        return len(self.smiles)

def make_collate_fn(vocab, use_bos_eos=True):
    bos = "<BOS>" if use_bos_eos else None
    eos = "<EOS>" if use_bos_eos else None
    def collate_fn(batch):
        smiles_batch, compound_id = zip(*batch)
        padded, pad_mask = collate_smiles_to_batch(list(smiles_batch), vocab,
                                                  pad_token="<PAD>", bos_token=bos, eos_token=eos)
        return padded, pad_mask, list(compound_id), list(smiles_batch)
    return collate_fn

In [5]:
# ============ 5. Positional Encoding ============
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=1000, learn_scale=True):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        pos = torch.arange(max_len, dtype=torch.float32).unsqueeze(1)
        div = torch.exp(torch.arange(0, d_model, 2, dtype=torch.float32) * (-math.log(10000.0) / d_model))
        pe[:, 0::2], pe[:, 1::2] = torch.sin(pos * div), torch.cos(pos * div)
        self.register_buffer("pe", pe.unsqueeze(1))
        self.alpha = nn.Parameter(torch.tensor(1.0)) if learn_scale else None

    def forward(self, x):
        pe = self.pe[:x.size(0)].to(dtype=x.dtype)
        return x + (self.alpha * pe if self.alpha is not None else pe)

In [6]:
# ============ 6. Transformer Classifier ============
class TransformerClassifier(nn.Module):
    def __init__(self, vocab_size, pos_encoder, nhead=4, d_model=128, dim_feedforward=512,
                 dropout=0.3, num_layers=2, use_cls_token=True, pad_idx=0, layerdrop=0.1):
        super().__init__()
        self.pool_weight_logit = nn.Parameter(torch.tensor(0.85))
        self.d_model = d_model
        self.use_cls = use_cls_token
        self.layerdrop = layerdrop
        self.pos_encoder=pos_encoder

        self.token_emb = nn.Embedding(vocab_size, d_model, padding_idx=pad_idx)
        self.emb_ln = nn.LayerNorm(d_model)
        self.emb_dropout = nn.Dropout(dropout)

        if self.use_cls:
            self.cls_token = nn.Parameter(torch.zeros(1, 1, d_model))
            nn.init.normal_(self.cls_token, mean=0.0, std=0.02)

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=nhead, dim_feedforward=dim_feedforward,
            dropout=dropout, activation="gelu", norm_first=True
        )
        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)

        self.mlp = nn.Sequential(
            nn.LayerNorm(d_model),
            nn.Linear(d_model, d_model // 2),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(d_model // 2, 1)
        )

    def forward(self, src, src_key_padding_mask):
        if src_key_padding_mask.dtype != torch.bool:
            src_key_padding_mask = src_key_padding_mask.bool()

        B, L = src.shape
        x = self.token_emb(src) * math.sqrt(self.d_model)
        x = x.transpose(0, 1)
        x = self.pos_encoder(x)
        x = self.emb_ln(x)
        x = self.emb_dropout(x)

        if self.use_cls:
            cls = self.cls_token.expand(1, B, -1).to(x.device)
            x = torch.cat([cls, x], dim=0)
            cls_mask = torch.zeros(B, 1, dtype=torch.bool, device=x.device)
            src_key_padding_mask = torch.cat([cls_mask, src_key_padding_mask], dim=1)
  
        for layer in self.encoder.layers:

            if self.training and torch.rand(1).item() < self.layerdrop:
                continue
            x = layer(x, src_key_padding_mask=src_key_padding_mask)

        memory = self.encoder.norm(x) if self.encoder.norm is not None else x
        
        #memory = self.encoder(x, src_key_padding_mask=src_key_padding_mask)
        if self.use_cls:
            cls_rep  = memory[0]  # CLS token pooling
            # cls_rep + mean_rep
            mem = memory[1:].transpose(0, 1)  # [B, L, d_model]
            mask = ~src_key_padding_mask[:, 1:mem.size(1)+1]
            denom = mask.sum(1, keepdim=True).clamp_min(1)
            mean_rep = (mem * mask.unsqueeze(-1)).sum(1) / denom
            # 加权融合
            weight = torch.sigmoid(self.pool_weight_logit)
            rep = weight * cls_rep + (1 - weight) * mean_rep
        else:
            mem = memory.transpose(0, 1)                     # [B, L, d_model]
            mask = ~src_key_padding_mask[:, :mem.size(1)]    # True=keep
            denom = mask.sum(1, keepdim=True).clamp_min(1)
            rep = (mem * mask.unsqueeze(-1)).sum(1) / denom  # masked mean pooling
        
        logits = self.mlp(rep).squeeze(-1)
        return logits

In [7]:
# ============ 7. VS  ============
def Virtual_Screening(model,data_loader,device):
    model.eval()
    compound_ids, compound_smiles, all_preds, all_probs = [], [], [], [] # [the number of all molecules in whole set]
    with torch.no_grad():
        for padded, pad_mask, compound_id, smiles in data_loader: # [B,L] [B,L] [B]
            # move to GPU
            padded = padded.to(device)
            pad_mask = pad_mask.to(device)
             
            logits = model(padded,pad_mask) # logits:[B] labels;[B]
            prob = torch.sigmoid(logits) # [B]
            pred = (prob > 0.5).float() # [B]

            compound_ids.extend(compound_id)
            compound_smiles.extend(smiles)
            all_probs.extend(prob.detach().cpu().numpy())
            all_preds.extend(pred.detach().cpu().numpy())
    
    # save values
    df = pd.DataFrame({
        "compound_id": compound_ids,
        "smiles": compound_smiles,
        "all_preds": all_preds,
        "all_probs": all_probs
    })

    df.to_csv(r"E:\Projects\Mycobacterium tuberculosis\05-VS\tcm_result.csv", index=False)
    return df

In [8]:
# 1) comfirm GPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(torch.cuda.get_device_name())

NVIDIA GeForce RTX 5070


In [9]:
# 2) load data
molecules = pd.read_csv(r"E:\Projects\Mycobacterium tuberculosis\05-VS\input\TCM_19076.csv")

In [10]:
molecules.head(1)

,tcm_id,smiles
0,34191.mol2Brassilexin,c1ccc2c(c1)[nH]c1sncc12


In [11]:
# 3) load vocab
with open(r"E:\Projects\Mycobacterium tuberculosis\05-VS\input\vocab.json", "r", encoding="utf-8") as f:
    vocab = json.load(f)

In [12]:
# 4) Prepare data processing tools
collate = make_collate_fn(vocab, use_bos_eos=True)
# 5) define the Dataset
dataset = SMILESDataset(molecules)
# 6) define the Dataloader
dataloader = DataLoader(dataset, batch_size=32, shuffle=False, collate_fn=collate)

In [13]:
# 7) Use best hyperparameters to define model
nhead=8
d_model=256
dim_feedforward=1024
dropout=0.1
num_layers=6
layerdrop=0.1

max_seq_len = 1000
pos_encoder = PositionalEncoding(d_model=d_model, max_len=max_seq_len, learn_scale=True).to(device)
model = TransformerClassifier(vocab_size=len(vocab), 
                              pos_encoder=pos_encoder,
                              nhead=nhead,
                              d_model=d_model,
                              dim_feedforward=dim_feedforward,
                              dropout=dropout,
                              num_layers=num_layers,
                              layerdrop=layerdrop).to(device)

In [14]:
# 8) set the optimizer and criterion
epochs = 100
lr = 0.0001
optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
criterion = nn.BCEWithLogitsLoss() # label must be float type
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)

model.load_state_dict(torch.load(
    r"E:\Projects\Mycobacterium tuberculosis\05-VS\input\transformer_classifier_8_best.pth",
    map_location=device  
))

<All keys matched successfully>

In [15]:
# 9) Virtual_Screening
results_df = Virtual_Screening(model,dataloader,device)